# Installing required packages

In [14]:
pip install beautifulsoup4 Requests pandas langdetect

     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 981.5/981.5 kB 5.8 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993362 sha256=44bdc4bb12b4556ba9b06a9af697723a27ca6ac2d93d33ff2e00a4eacfa9ff3e
  Stored in directory: c:\users\dj140\appdata\local\packages\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\localcache\local\pip\cache\wheels\0a\f2\b2\e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
Note: you may need to restart the kernel to use up

# Importing required libraries

In [16]:
from bs4 import BeautifulSoup
from pathlib import Path
import requests, csv, json, time
import pandas as pd
from langdetect import detect, DetectorFactory
import re

In [4]:
main_dir = Path.cwd()
print(main_dir)

c:\Users\dj140\OneDrive\Desktop\project\MSc-DSA-2026-Thesis-Project\WebScraper


# Validate if valid sentences

In [19]:
DetectorFactory.seed = 42

def is_english(text, threshold=0.5):

    if not isinstance(text, str):
        return False
    
    if not text or len(text.strip()) == 0:
        return False

    try:
        lang = detect(text)
        return lang == 'en'
    except:
        return is_english_chars(text, threshold)

def is_english_chars(text, threshold=0.5):

    if not text or len(text.strip()) == 0:
        return False
    
    english_chars = re.findall(r'[a-zA-Z0-9\s\.\,\!\?\'\"]', text)
    ratio = len(english_chars) / len(text)
    return ratio >= threshold

# Initial values for Steam Data

In [7]:
FORUM_LIST_PAGE_SIZE_STEAM = 10
COMMENT_LIST_PAGE_SIZE_STEAM = 6
rows_steam = []
id_value_steam = 1

# Steam Web Scraping Process

In [8]:
with open(main_dir / "games_list" / "steam_urls.json", "r", encoding="utf-8") as f:
    game_urls = json.load(f)

print(game_urls)

{'Counter-Strike 2': 'https://steamcommunity.com/app/730/discussions/0/', 'Helldivers 2': 'https://steamcommunity.com/app/553850/discussions/0/', 'Dota 2': 'https://steamcommunity.com/app/570/discussions/0/', 'Team Fortress 2': 'https://steamcommunity.com/app/440/discussions/0/'}


In [9]:
for game, url in game_urls.items():

    print("Scrapping Text for game: " + game + "...")

    for i in range(FORUM_LIST_PAGE_SIZE_STEAM):

        params = {
                "fp" : (i + 1)
            }

        data = requests.get(url, params=params)
        
        parent_html = data.text

        soup = BeautifulSoup(parent_html, 'lxml')

        forum_discussions = soup.find('div', class_='forum_area').find_all('a', class_='forum_topic_overlay')

        for discussion in forum_discussions:

            for j in range(COMMENT_LIST_PAGE_SIZE_STEAM):
                child_params = {
                                "ctp" : (j + 1)
                            }

                new_resp = requests.get(discussion['href'], params=child_params)

                new_data = new_resp.text

                child_soup = BeautifulSoup(new_data, 'lxml')

                comments = child_soup.find_all('div', class_='commentthread_comment_text')

                for comment in comments:

                    for child in comment.find_all():
                        child.extract()

                    text = comment.get_text(strip=True)
                    rows_steam.append({
                        "id": id_value_steam,
                        "game": game,
                        "text": text
                    })
                    id_value_steam = id_value_steam + 1


    print('Completed\n')

Scrapping Text for game: Counter-Strike 2...
Completed

Scrapping Text for game: Helldivers 2...
Completed

Scrapping Text for game: Dota 2...
Completed

Scrapping Text for game: Team Fortress 2...
Completed



In [10]:
with open(main_dir / "text_data" / "steam_text_data.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "game", "text"])
    writer.writeheader()
    writer.writerows(rows_steam)

In [20]:
temp_df = pd.read_csv(main_dir / "text_data" / "steam_text_data.csv")

temp_df = temp_df[temp_df["text"].notna() & (temp_df["text"].str.strip() != '') & temp_df['text'].apply(is_english)]

output_dir = main_dir / "text_data" / "Steam"

for game_name, group_df in temp_df.groupby('game'):

    filename = "".join(c for c in game_name if c.isalnum() or c in (' ', '-', '_')).strip().replace(' ', '_')
    output_path = output_dir / f"{filename}.csv"
    group_df[["text"]].to_csv(output_path, index=False, encoding='utf-8')

# Initial values for Reddit Data

In [29]:
SUBREDDIT_LIST_SIZE = 3
COMMENT_LIST_SIZE_REDDIT = 2
rows_reddit = []
id_value_reddit = 1

# Reddit Web Scraping Process

In [14]:
with open(main_dir / "games_list" / "reddit_urls.json", "r", encoding="utf-8") as f:
    game_urls = json.load(f)

print(game_urls)

{'League of Legends': 'https://www.reddit.com/r/leagueoflegends/new.rss'}


In [30]:
headers = {'User-agent': 'RedditScraper/0.1'}

params = {
    "limit": SUBREDDIT_LIST_SIZE
}

In [ ]:
for game, url in game_urls.items():

    print("Scrapping Text for game: " + game + "...")

    data = requests.get(url, headers=headers, params=params)

    # print(data.text)

    soup = BeautifulSoup(data.text, 'xml')

    items = soup.find_all('entry')

    # print(items)

    for item in items:
        item_url = item.find('link')['href'][:-1] + '.rss'
        print(item_url)
        time.sleep(120)
        item_data = requests.get(item_url, headers=headers)
        item_soup = BeautifulSoup(item_data.text, 'xml')
        
        comments = item_soup.find_all('entry')

        # print(item_soup.find_all('entry'))

        for comment in comments:
            # print(comment.find('content').string)
            html_elem = BeautifulSoup(comment.find('content').string.
                                      replace('<!-- SC_OFF -->', '').
                                      replace('<!-- SC_ON -->', ''), "lxml")
            
            text = html_elem.find('div', class_='md').get_text()

            # print(text)

            rows_reddit.append({
                        "id": id_value_reddit,
                        "game": game,
                        "text": text
                    })
            id_value_reddit = id_value_reddit + 1


print('Completed\n')

Scrapping Text for game: League of Legends...
https://www.reddit.com/r/leagueoflegends/comments/1uv8aax/bausffs_shares_his_brutally_honest_take_on.rss
https://www.reddit.com/r/leagueoflegends/comments/1uv7kbw/i_think_zeka_is_so_fun_to_watch.rss
https://www.reddit.com/r/leagueoflegends/comments/1uv6wof/whats_bannable.rss
Completed



In [34]:
with open(main_dir / "text_data" / "reddit_text_data.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "game", "text"])
    writer.writeheader()
    writer.writerows(rows_reddit)